# Act, Ask, or Defer — ManiSkill run (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LetsTrie/Uncertainty-Aware-Robot-Simulation/blob/main/notebooks/run_maniskill_colab.ipynb)

Runs the **Uncertainty-Aware Selective Autonomy** project on a real
[ManiSkill](https://maniskill.readthedocs.io) robot scene, on Colab's free GPU.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

Why the extra setup below: Colab ships **Python 3.13**, but ManiSkill's
dependencies (`sapien`, `mplib`) have no 3.13 wheels — so we build an isolated
**Python 3.11** with `uv` and run everything through it. SAPIEN also needs a
**Vulkan** driver file, which we add. Total time: ~5 minutes.

## 1 · Clone the repository

In [ ]:
!git clone https://github.com/LetsTrie/Uncertainty-Aware-Robot-Simulation.git
%cd Uncertainty-Aware-Robot-Simulation

## 2 · Build an isolated Python 3.11 with `uv`
`uv` downloads a standalone 3.11 that ignores Colab's 3.13.

In [ ]:
!pip install -q uv
!uv venv --python 3.11 /content/ms
!/content/ms/bin/python --version   # should print 3.11.x

## 3 · Install the dependencies into that 3.11 env
Pinned to `mani_skill==3.0.1` so pip resolves instantly.

In [ ]:
!uv pip install --python /content/ms/bin/python \
    "mani_skill==3.0.1" torch numpy pandas matplotlib pyyaml

## 4 · Set up Vulkan for the SAPIEN renderer
Colab's GPU driver includes `libGLX_nvidia.so.0`; we point Vulkan at it.

In [ ]:
%%bash
apt-get -qq install -y libvulkan1 > /dev/null
mkdir -p /usr/share/vulkan/icd.d
cat > /usr/share/vulkan/icd.d/nvidia_icd.json <<'EOF'
{ "file_format_version": "1.0.0",
  "ICD": { "library_path": "libGLX_nvidia.so.0", "api_version": "1.3.277" } }
EOF

## 5 · Verify ManiSkill renders on the GPU
Should print `ManiSkill GPU OK`.

In [ ]:
!/content/ms/bin/python -c "import gymnasium as gym, mani_skill.envs; e=gym.make('PickCube-v1', num_envs=1, obs_mode='state'); e.reset(); print('ManiSkill GPU OK'); e.close()"

## 6 · Generate the dataset and train the policies
The episode CSVs are not stored in the repo, so we regenerate them here.

In [ ]:
!/content/ms/bin/python scripts/generate_dataset.py
!/content/ms/bin/python scripts/train_policy.py

## 7 · Run the ManiSkill sweep
Evaluates every policy on `train` (3 cubes), `shift` (3 cubes), and `shift` (4 cubes) — 500 episodes each — and writes `results/tables/maniskill_results.csv`.

In [ ]:
!/content/ms/bin/python scripts/collect_maniskill.py --n 500

## 8 · Look at a rendered robot scene

In [ ]:
from IPython.display import Image
Image('results/plots/maniskill_frames/shift_c4/episode_00.png')

## 9 · Download the results

In [ ]:
from google.colab import files
files.download('results/tables/maniskill_results.csv')